# 05 — Arquitecturas Mixtas (Conv1D + LSTM/GRU)
Combina extracción de patrones locales (Conv1D) con memoria temporal (LSTM/GRU).
Input `(N, V_in, 23)`. Catálogo activo: **conv_lstm**. Descomentar `[EXTENDER]` para más.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, yfinance as yf
warnings.simplefilter('ignore')

from keras import Sequential, Input
from keras.layers import (Conv1D, GlobalAveragePooling1D,
                          LSTM, GRU, Dense)

from utils import (TICKERS, INPUT_WINDOWS, OUTPUT_WINDOWS,
                   create_time_series_data, make_splits, eval_mae,
                   get_callbacks, compile_model,
                   plot_mae_matrix, build_results_df, best_per_window)

In [ ]:
# ── HIPERPARÁMETROS ───────────────────────────────────────────
EPOCHS     = 100
BATCH_SIZE = 64
QUICK_MODE = False
if QUICK_MODE: EPOCHS = 20

precios = yf.download(TICKERS, start='1945-01-01', auto_adjust=True, progress=False)['Close']
precios.dropna(axis=1, inplace=True)
returns = np.log(precios).diff().dropna()
print(f'Retornos: {returns.shape}  |  EPOCHS={EPOCHS}')

## Catálogo de arquitecturas mixtas
Conv1D filtra patrones locales; LSTM/GRU captura dependencias de largo plazo sobre los mapas de características resultantes.

In [ ]:
MODELOS = {
    # Conv1D → LSTM: patrones locales + memoria secuencial
    'conv_lstm': lambda V: compile_model(Sequential([
        Input((V, 23)),
        Conv1D(64, kernel_size=3, activation='relu'),
        LSTM(64),
        Dense(23)])),

    # 'conv_gru': lambda V: compile_model(Sequential([        # [EXTENDER]
    #     Input((V, 23)),
    #     Conv1D(64, kernel_size=3, activation='relu'),
    #     GRU(64),
    #     Dense(23)])),

    # 'conv_dense': lambda V: compile_model(Sequential([      # [EXTENDER] sin recurrencia
    #     Input((V, 23)),
    #     Conv1D(64, kernel_size=3, activation='relu'),
    #     GlobalAveragePooling1D(),
    #     Dense(64, activation='relu'),
    #     Dense(23)])),

    # 'lstm_dense': lambda V: compile_model(Sequential([      # [EXTENDER] LSTM + MLP
    #     Input((V, 23)),
    #     LSTM(64),
    #     Dense(64, activation='relu'),
    #     Dense(23)])),
}

## Entrenamiento

In [ ]:
results     = {}
historiales = {}

for nombre, build_fn in MODELOS.items():
    for V_in in INPUT_WINDOWS:
        for V_out in OUTPUT_WINDOWS:
            X, y = create_time_series_data(returns, V_in, V_out)
            X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)

            model = build_fn(V_in)
            hist  = model.fit(X_tr, y_tr,
                              validation_data=(X_v, y_v),
                              epochs=EPOCHS, batch_size=BATCH_SIZE,
                              callbacks=get_callbacks(), verbose=0)

            key = (nombre, V_in, V_out)
            results[key]     = {'train':  eval_mae(model, X_tr, y_tr),
                                'val':    eval_mae(model, X_v,  y_v),
                                'test':   eval_mae(model, X_ts, y_ts),
                                'params': model.count_params()}
            historiales[key] = hist
            print(f'{nombre}  in={V_in:2d}  out={V_out:2d}  '
                  f'train={results[key]["train"]:.4f}  '
                  f'val={results[key]["val"]:.4f}  '
                  f'test={results[key]["test"]:.4f}  '
                  f'ep={len(hist.history["loss"])}')

## Curvas de convergencia

In [ ]:
for nombre in MODELOS:
    fig, axes = plt.subplots(4, 4, figsize=(16, 12))
    for i, V_in in enumerate(INPUT_WINDOWS):
        for j, V_out in enumerate(OUTPUT_WINDOWS):
            hist = historiales[(nombre, V_in, V_out)]
            ax = axes[i][j]
            ax.plot(hist.history['loss'],     label='train')
            ax.plot(hist.history['val_loss'], label='val')
            ax.set_title(f'in={V_in} out={V_out}', fontsize=8)
            ax.legend(fontsize=6); ax.tick_params(labelsize=6)
    plt.suptitle(f'Convergencia — {nombre}', fontsize=12)
    plt.tight_layout(); plt.show()

## Resultados

In [ ]:
df_res = build_results_df(results)
print(df_res.to_string())

mat = best_per_window(df_res, metric='test')
plot_mae_matrix(mat, title='Mixtos — mejor MAE test')